# 13 · App-wide plugins and usage

A plugin applies cross-cutting behavior across all agents handled by a Runner. Keep logs limited to synthetic metadata and token counts.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** Lite model calls. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from google.adk.plugins.base_plugin import BasePlugin

class UsagePlugin(BasePlugin):
    def __init__(self):
        super().__init__(name="course_usage")
        self.calls = 0
        self.tokens = 0

    async def before_model_callback(self, *, callback_context, llm_request):
        self.calls += 1
        print("Model request", self.calls, "agent", callback_context.agent_name)

    async def after_model_callback(self, *, callback_context, llm_response):
        if llm_response.usage_metadata:
            self.tokens += llm_response.usage_metadata.total_token_count or 0

plugin = UsagePlugin()
lab = await make_lab(llm("observed_shop", "Use tools to answer order questions.", tools=[get_order]), plugins=[plugin])
events = await ask(lab, "What is O1001's total?")
print({"model_requests": plugin.calls, "reported_tokens": plugin.tokens})
print([{ "author": e.author, "event_id": e.id, "invocation_id": e.invocation_id,
         "tools": [c.name for c in e.get_function_calls()] } for e in events])


## Try it

Add a before_tool_callback on the plugin to count tool calls. ADK also supports OpenTelemetry exporters; no paid telemetry backend is required here.


## Reference

[Official ADK documentation](https://adk.dev/plugins/). Shared configuration: `config.json`. No environment activation or login cells are needed.
